# Transcribe Stacey Boehman Live Coaching Calls
**Instructions:**
1. Upload your MP3 files to Google Drive in a folder called `coaching_audio`
2. Run each cell in order
3. Transcripts will be saved to Google Drive in `coaching_transcripts/`
4. Download the `.md` files and upload to Claude Projects

In [ ]:
# Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Install faster-whisper
!pip install faster-whisper -q

In [ ]:
# Cell 3: Transcribe all MP3s
import time
from pathlib import Path
from faster_whisper import WhisperModel

MP3_DIR        = Path('/content/drive/MyDrive/coaching_audio')
TRANSCRIPT_DIR = Path('/content/drive/MyDrive/coaching_transcripts')
TRANSCRIPT_DIR.mkdir(exist_ok=True)

# large-v3 on Colab GPU: ~2 min per hour of audio
print('Loading Whisper large-v3 on GPU...')
model = WhisperModel('large-v3', device='cuda', compute_type='float16')
print('Model loaded.\n')

mp3s = sorted(MP3_DIR.glob('*.mp3'))
print(f'Found {len(mp3s)} MP3s in {MP3_DIR}\n')

for i, mp3 in enumerate(mp3s, 1):
    out = TRANSCRIPT_DIR / (mp3.stem + '.md')
    if out.exists():
        print(f'[{i}/{len(mp3s)}] SKIP (already done): {mp3.name}')
        continue
    print(f'[{i}/{len(mp3s)}] {mp3.name}')
    t0 = time.time()
    segments, info = model.transcribe(str(mp3), language='en', beam_size=5)
    text = ' '.join(s.text.strip() for s in segments)
    title = mp3.stem.replace('_', ' ').strip()
    out.write_text(f'# {title}\n\n---\n\n{text}\n', encoding='utf-8')
    elapsed = time.time() - t0
    print(f'  Done in {elapsed:.0f}s — {out.stat().st_size//1024}KB saved to Drive')
    print()

print(f'\nAll done! Transcripts saved to {TRANSCRIPT_DIR}')